In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.layers import Layer, Dense, LayerNormalization, Dropout, LSTM, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
import numpy as np
import pandas as pd

#create a synthetic stock pricedataset
np.random.seed(42)
data_length = 2000
trend = np.linspace(100,200,data_length)
noise = np.random.normal(0,10,data_length)
synthetic_data = trend + noise

# Create a dataframe and save to CSV
data = pd.DataFrame(synthetic_data,columns=['Price'])
data.to_csv('synthetic_stock_data.csv',index=False)
print("Synthetic stock data saved to synthetic_stock_data.csv")



Synthetic stock data saved to synthetic_stock_data.csv


In [3]:
# Load the dataset
data = pd.read_csv('synthetic_stock_data.csv')
data = data['Price'].values

# Normalize the data (MinMaxScaler expects 2D input: samples x features)
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data.reshape(-1, 1))

# Set FAST_MODE=True for quicker runs; False for fuller experiments
FAST_MODE = True

TRAIN_CONFIG = {
    'epochs': 12 if FAST_MODE else 50,
    'batch_size': 128,
    'early_stop_patience': 3 if FAST_MODE else 5,
    'lr_patience': 2 if FAST_MODE else 3,
    'time_step': 60 if FAST_MODE else 100,
    'time_step_options': [40, 60] if FAST_MODE else [50, 100, 150],
    'window_compare_epochs': 8 if FAST_MODE else 30,
    'embed_dim': 32 if FAST_MODE else 64,
    'num_heads': 2 if FAST_MODE else 4,
    'ff_dim': 64 if FAST_MODE else 128,
    'num_layers': 1 if FAST_MODE else 2,
    'lstm_units': 32 if FAST_MODE else 64,
    'max_train_samples': 1000 if FAST_MODE else None,
}

def create_dataset(data, time_step=1):
    X, Y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        Y.append(data[i + time_step, 0])
    return np.array(X), np.array(Y)

def prepare_sequences(time_step):
    X, Y = create_dataset(data_scaled, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    return X, Y

def time_ordered_split(X, Y, split_ratio=0.8, max_train_samples=None):
    split = int(len(X) * split_ratio)
    X_train, X_test = X[:split], X[split:]
    Y_train, Y_test = Y[:split], Y[split:]
    if max_train_samples is not None:
        X_train, Y_train = X_train[:max_train_samples], Y_train[:max_train_samples]
    return X_train, X_test, Y_train, Y_test

def inverse_transform_predictions(preds, targets):
    preds = scaler.inverse_transform(np.asarray(preds).reshape(-1, 1))
    targets = scaler.inverse_transform(np.asarray(targets).reshape(-1, 1))
    return preds.flatten(), targets.flatten()

def evaluate_predictions(preds, targets):
    return {
        'mse': mean_squared_error(targets, preds),
        'mae': mean_absolute_error(targets, preds),
        'rmse': np.sqrt(mean_squared_error(targets, preds)),
    }

time_step = TRAIN_CONFIG['time_step']
X, Y = prepare_sequences(time_step)
print(f"FAST_MODE: {FAST_MODE}")
print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)

FAST_MODE: True
Shape of X: (1939, 60, 1)
Shape of Y: (1939,)


In [4]:
class MultiHeadSelfAttention(Layer): 

    def __init__(self, embed_dim, num_heads=8): 
        super(MultiHeadSelfAttention, self).__init__() 
        self.embed_dim = embed_dim 
        self.num_heads = num_heads 
        self.projection_dim = embed_dim // num_heads 
        self.query_dense = Dense(embed_dim) 
        self.key_dense = Dense(embed_dim) 
        self.value_dense = Dense(embed_dim) 
        self.combine_heads = Dense(embed_dim) 


    def attention(self, query, key, value): 
        score = tf.matmul(query, key, transpose_b=True) 
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32) 
        scaled_score = score / tf.math.sqrt(dim_key) 
        weights = tf.nn.softmax(scaled_score, axis=-1) 
        output = tf.matmul(weights, value) 
        return output, weights 

    def split_heads(self, x, batch_size): 
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim)) 
        return tf.transpose(x, perm=[0, 2, 1, 3]) 

    def call(self, inputs): 
        batch_size = tf.shape(inputs)[0] 
        query = self.query_dense(inputs) 
        key = self.key_dense(inputs) 
        value = self.value_dense(inputs) 
        query = self.split_heads(query, batch_size) 
        key = self.split_heads(key, batch_size) 
        value = self.split_heads(value, batch_size) 
        attention, _ = self.attention(query, key, value) 
        attention = tf.transpose(attention, perm=[0, 2, 1, 3]) 
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim)) 
        output = self.combine_heads(concat_attention) 
        return output 

 

In [5]:
class TransformerBlock(Layer): 

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1): 
        super(TransformerBlock, self).__init__() 
        self.att = MultiHeadSelfAttention(embed_dim, num_heads) 
        self.ffn = tf.keras.Sequential([ 
            Dense(ff_dim, activation="relu"), 
            Dense(embed_dim), 
        ]) 

        self.layernorm1 = LayerNormalization(epsilon=1e-6) 
        self.layernorm2 = LayerNormalization(epsilon=1e-6) 
        self.dropout1 = Dropout(rate) 
        self.dropout2 = Dropout(rate) 


    def call(self, inputs, training): 
        attn_output = self.att(inputs) 
        attn_output = self.dropout1(attn_output, training=training) 
        out1 = self.layernorm1(inputs + attn_output) 
        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training) 
        return self.layernorm2(out1 + ffn_output) 

In [6]:
class EncoderLayer(Layer): 

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1): 
        super(EncoderLayer, self).__init__() 
        self.att = MultiHeadSelfAttention(embed_dim, num_heads) 
        self.ffn = tf.keras.Sequential([ 
            Dense(ff_dim, activation="relu"), 
            Dense(embed_dim), 
        ]) 

        self.layernorm1 = LayerNormalization(epsilon=1e-6) 
        self.layernorm2 = LayerNormalization(epsilon=1e-6) 
        self.dropout1 = Dropout(rate) 
        self.dropout2 = Dropout(rate) 

 

    def call(self, inputs, training): 
        attn_output = self.att(inputs) 
        attn_output = self.dropout1(attn_output, training=training) 
        out1 = self.layernorm1(inputs + attn_output) 
        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training) 
        return self.layernorm2(out1 + ffn_output) 



In [7]:
import tensorflow as tf 
from tensorflow.keras.layers import Layer, Dense, LayerNormalization, Dropout 

class MultiHeadSelfAttention(Layer): 
    def __init__(self, embed_dim, num_heads=8): 
        super(MultiHeadSelfAttention, self).__init__() 
        self.embed_dim = embed_dim 
        self.num_heads = num_heads 
        self.projection_dim = embed_dim // num_heads 
        self.query_dense = Dense(embed_dim) 
        self.key_dense = Dense(embed_dim) 
        self.value_dense = Dense(embed_dim) 
        self.combine_heads = Dense(embed_dim) 
 

    def attention(self, query, key, value): 
        score = tf.matmul(query, key, transpose_b=True) 
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32) 
        scaled_score = score / tf.math.sqrt(dim_key) 
        weights = tf.nn.softmax(scaled_score, axis=-1) 
        output = tf.matmul(weights, value) 
        return output, weights 


    def split_heads(self, x, batch_size): 
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim)) 
        return tf.transpose(x, perm=[0, 2, 1, 3]) 


    def call(self, inputs): 
        batch_size = tf.shape(inputs)[0] 
        query = self.query_dense(inputs) 
        key = self.key_dense(inputs) 
        value = self.value_dense(inputs) 
        query = self.split_heads(query, batch_size) 
        key = self.split_heads(key, batch_size) 
        value = self.split_heads(value, batch_size) 
        attention, _ = self.attention(query, key, value) 
        attention = tf.transpose(attention, perm=[0, 2, 1, 3]) 
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim)) 
        output = self.combine_heads(concat_attention) 
        return output 

class TransformerBlock(Layer): 
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1): 
        super(TransformerBlock, self).__init__() 
        self.att = MultiHeadSelfAttention(embed_dim, num_heads) 
        self.ffn = tf.keras.Sequential([ 
            Dense(ff_dim, activation="relu"), 
            Dense(embed_dim), 
        ]) 

        self.layernorm1 = LayerNormalization(epsilon=1e-6) 
        self.layernorm2 = LayerNormalization(epsilon=1e-6) 
        self.dropout1 = Dropout(rate) 
        self.dropout2 = Dropout(rate) 
 

    def call(self, inputs, training): 
        attn_output = self.att(inputs) 
        attn_output = self.dropout1(attn_output, training=training) 
        out1 = self.layernorm1(inputs + attn_output) 
        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training) 
        return self.layernorm2(out1 + ffn_output) 

class TransformerEncoder(Layer): 
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1): 
        super(TransformerEncoder, self).__init__() 
        self.num_layers = num_layers 
        self.embed_dim = embed_dim 
        self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate) for _ in range(num_layers)] 
        self.dropout = Dropout(rate) 

    def call(self, inputs, training=False): 
        x = inputs 
        for i in range(self.num_layers): 
            x = self.enc_layers[i](x, training=training) 
        return x 

# Example usage 
embed_dim = 128 
num_heads = 8 
ff_dim = 512 
num_layers = 4 

transformer_encoder = TransformerEncoder(num_layers, embed_dim, num_heads, ff_dim) 
inputs = tf.random.uniform((1, 100, embed_dim)) 
outputs = transformer_encoder(inputs, training=False)  # Use keyword argument for 'training' 
print(outputs.shape)  # Should print (1, 100, 128) 

(1, 100, 128)


In [8]:
def build_training_callbacks():
    return [
        EarlyStopping(
            monitor='val_loss',
            patience=TRAIN_CONFIG['early_stop_patience'],
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=TRAIN_CONFIG['lr_patience'],
            min_lr=1e-6,
            verbose=1,
        ),
    ]

def train_model(model, X_train, Y_train, X_val, Y_val, epochs=None, verbose=1):
    return model.fit(
        X_train, Y_train,
        epochs=epochs or TRAIN_CONFIG['epochs'],
        batch_size=TRAIN_CONFIG['batch_size'],
        validation_data=(X_val, Y_val),
        callbacks=build_training_callbacks(),
        verbose=verbose,
    )

def build_transformer_model(
    input_shape,
    embed_dim=None,
    num_heads=None,
    ff_dim=None,
    num_layers=None,
):
    embed_dim = embed_dim or TRAIN_CONFIG['embed_dim']
    num_heads = num_heads or TRAIN_CONFIG['num_heads']
    ff_dim = ff_dim or TRAIN_CONFIG['ff_dim']
    num_layers = num_layers or TRAIN_CONFIG['num_layers']

    inputs = tf.keras.Input(shape=input_shape)
    x = Dense(embed_dim)(inputs)
    encoder_outputs = TransformerEncoder(num_layers, embed_dim, num_heads, ff_dim)(x)
    pooled = GlobalAveragePooling1D()(encoder_outputs)
    outputs = Dense(1)(pooled)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

def build_lstm_model(input_shape, units=None):
    units = units or TRAIN_CONFIG['lstm_units']
    model = tf.keras.Sequential([
        LSTM(units, input_shape=input_shape),
        Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

input_shape = (X.shape[1], X.shape[2])
model = build_transformer_model(input_shape)
model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 60, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 60, 32)         │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_1           │ (None, 60, 32)         │         8,544 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,641 (33.75 KB)

 Trainable params: 8,641 (33.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train with early stopping and learning-rate scheduling
X_train, X_test, Y_train, Y_test = time_ordered_split(
    X, Y, max_train_samples=TRAIN_CONFIG['max_train_samples']
)

print(f"Training on {len(X_train)} samples, validating on {len(X_test)} samples")
history = train_model(model, X_train, Y_train, X_test, Y_test)

Training on 1000 samples, validating on 388 samples
Epoch 1/12


In [ ]:
# Evaluate Transformer on the test set
transformer_preds_scaled = model.predict(X_test, batch_size=128, verbose=1)
transformer_preds, transformer_true = inverse_transform_predictions(transformer_preds_scaled, Y_test)
transformer_metrics = evaluate_predictions(transformer_preds, transformer_true)

print("Transformer test metrics:")
for metric, value in transformer_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

true_values = scaler.inverse_transform(data.reshape(-1, 1))
split = len(X_train)

plt.figure(figsize=(12, 5))
plt.plot(true_values, label='True Data', alpha=0.7)
test_start = time_step + split
plt.plot(
    np.arange(test_start, test_start + len(transformer_preds)),
    transformer_preds,
    label='Transformer Predictions',
)
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()
plt.title('Transformer: Test Predictions vs True Data')
plt.show()

In [ ]:
# Compare different lookback windows (time_step)
TIME_STEP_OPTIONS = TRAIN_CONFIG['time_step_options']
time_step_results = []

for ts in TIME_STEP_OPTIONS:
    X_ts, Y_ts = prepare_sequences(ts)
    X_tr, X_te, Y_tr, Y_te = time_ordered_split(
        X_ts, Y_ts, max_train_samples=TRAIN_CONFIG['max_train_samples']
    )

    window_model = build_transformer_model((ts, 1))
    train_model(
        window_model, X_tr, Y_tr, X_te, Y_te,
        epochs=TRAIN_CONFIG['window_compare_epochs'],
        verbose=0,
    )

    preds_scaled = window_model.predict(X_te, batch_size=TRAIN_CONFIG['batch_size'], verbose=0)
    preds, true = inverse_transform_predictions(preds_scaled, Y_te)
    metrics = evaluate_predictions(preds, true)
    time_step_results.append({'time_step': ts, **metrics})
    print(f"time_step={ts}: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}")

time_step_df = pd.DataFrame(time_step_results).set_index('time_step')
print("\nTime-step comparison:")
print(time_step_df.round(4))

best_time_step = time_step_df['rmse'].idxmin()
print(f"\nBest window by RMSE: {best_time_step}")

In [ ]:
# Baseline models on the same train/test split
lstm_model = build_lstm_model(input_shape)
train_model(lstm_model, X_train, Y_train, X_test, Y_test)

lstm_preds_scaled = lstm_model.predict(X_test, batch_size=TRAIN_CONFIG['batch_size'], verbose=0)
lstm_preds, lstm_true = inverse_transform_predictions(lstm_preds_scaled, Y_test)
lstm_metrics = evaluate_predictions(lstm_preds, lstm_true)

# Naive baseline: predict the last observed price in each window
naive_preds_scaled = X_test[:, -1, 0]
naive_preds, naive_true = inverse_transform_predictions(naive_preds_scaled, Y_test)
naive_metrics = evaluate_predictions(naive_preds, naive_true)

print("LSTM test metrics:")
for metric, value in lstm_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

print("\nNaive baseline test metrics:")
for metric, value in naive_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

In [ ]:
# Compare Transformer, LSTM, and naive baseline
comparison_df = pd.DataFrame([
    {'model': 'Transformer', **transformer_metrics},
    {'model': 'LSTM', **lstm_metrics},
    {'model': 'Naive (last price)', **naive_metrics},
]).set_index('model')

print("Model comparison (lower is better):")
print(comparison_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparison_df[['rmse', 'mae']].plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title('Test Error by Model')
axes[0].set_ylabel('Error')
axes[0].legend()

test_index = np.arange(test_start, test_start + len(transformer_preds))
axes[1].plot(test_index, transformer_true, label='True', alpha=0.7)
axes[1].plot(test_index, transformer_preds, label='Transformer')
axes[1].plot(test_index, lstm_preds, label='LSTM')
axes[1].plot(test_index, naive_preds, label='Naive', linestyle='--')
axes[1].set_title('Test Set Predictions')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Stock Price')
axes[1].legend()

plt.tight_layout()
plt.show()